# Chapter 11: Binary Heaps and Heap Sort

A progressive guide to representing heaps, keeping their priority rule intact, and using them to sort and select values.

**Learning goals**
- Explain the shape and ordering rules of a binary heap.
- Implement a comparator-based min heap and max heap.
- Build a heap efficiently and sort an array with heap sort.
- Use heaps for a task queue and top-k selection.

The examples use plain JavaScript and require no external packages.

## Running this notebook

Select the **JavaScript (Node.js)** kernel and run the cells from top to bottom. A binary heap is a **complete** binary tree whose parent-child order is constrained: in a min heap each parent is no greater than its children; in a max heap each parent is no less than its children. The root is therefore the next minimum or maximum. Siblings are not necessarily ordered, so a heap array is not a sorted array.

## 1. Understand the two heap rules

A binary heap is a complete binary tree: every level is full except possibly the last, which fills from left to right without gaps. It must also follow an ordering rule. A min heap has each parent less than or equal to its children; a max heap has each parent greater than or equal to its children. Therefore, the root holds the minimum or maximum value, respectively.

A heap is useful for a priority queue, such as a scheduler that repeatedly selects the most urgent task. Siblings have no required order, so a heap array is not generally sorted. Unlike a binary search tree, a heap only constrains each parent relative to its children.

**Try it in Notebook**

Is `[2, 6, 3, 5]` a min heap? Identify the parent-child pair that decides your answer.

In [ ]:
function checkMinHeap(values) {
    for (let child = 1; child < values.length; child++) {
        const parent = Math.floor((child - 1) / 2);
        if (values[parent] > values[child]) {
            return {
                isMinHeap: false,
                parent: { index: parent, value: values[parent] },
                child: { index: child, value: values[child] }
            };
        }
    }
    return { isMinHeap: true, parent: null, child: null };
}

const firstTryIt = checkMinHeap([2, 6, 3, 5]);
console.log('Is a min heap:', firstTryIt.isMinHeap);
console.log('Violating parent-child pair:', firstTryIt.parent, firstTryIt.child);

## 2. Store a complete tree in an array

A complete tree fills each level from left to right. Store its nodes in level order, and the parent/child relationships follow from the index. With zero-based indexes, the left child of `i` is `2 * i + 1`, the right child is `2 * i + 2`, and a non-root node's parent is `Math.floor((i - 1) / 2)`. A child exists only when its index is less than the array length.

In [ ]:
const heapArrayExample = [2, 3, 4, 5, 8, 7, 9];
const heapIndex = 2;
console.log({
    value: heapArrayExample[heapIndex],
    parentIndex: Math.floor((heapIndex - 1) / 2),
    leftChildIndex: 2 * heapIndex + 1,
    rightChildIndex: 2 * heapIndex + 2
});
console.log('Index 3 children exist:', 7 < heapArrayExample.length);

**Try it in Notebook**

In an array of length 7, find the parent and child indexes of index 2. Then find the child indexes of index 3 and decide whether those children exist.

In [ ]:
const arrayLength = 7;
const indexToCheck = 2;
const indexTwoRelationships = {
    parent: Math.floor((indexToCheck - 1) / 2),
    leftChild: 2 * indexToCheck + 1,
    rightChild: 2 * indexToCheck + 2
};
const indexThreeChildren = [2 * 3 + 1, 2 * 3 + 2];
const indexThreeExistence = indexThreeChildren.map(child => child < arrayLength);
console.log('Index 2 relationships:', indexTwoRelationships);
console.log('Index 3 children:', indexThreeChildren);
console.log('Do they exist?', indexThreeExistence);

## 3. Insert by sifting up

Append the new value to preserve the complete shape. If it has higher priority than its parent, swap them and repeat toward the root. Only the new node's path can violate the heap order. The repair follows one path of logarithmic height, so insertion is **O(log n)** in the worst case.

**Try it in Notebook**

Insert `2` into `[3, 5, 4, 9, 7]`. Write the array after appending and after each swap.

In [ ]:
const insertionTryHeap = [3, 5, 4, 9, 7];
const insertionTrySteps = [];
insertionTryHeap.push(2);
insertionTrySteps.push({ step: 'Append 2', heap: insertionTryHeap.slice() });

let insertionTryIndex = insertionTryHeap.length - 1;
while (insertionTryIndex > 0) {
    const parentIndex = Math.floor((insertionTryIndex - 1) / 2);
    if (insertionTryHeap[insertionTryIndex] >= insertionTryHeap[parentIndex]) break;
    [insertionTryHeap[insertionTryIndex], insertionTryHeap[parentIndex]] =
        [insertionTryHeap[parentIndex], insertionTryHeap[insertionTryIndex]];
    insertionTrySteps.push({
        step: `Swap 2 with parent ${insertionTryHeap[insertionTryIndex]}`,
        heap: insertionTryHeap.slice()
    });
    insertionTryIndex = parentIndex;
}
console.log(insertionTrySteps);

## 4. Implement a min heap

The comparator returns a negative number when its first argument has higher priority. The default comparator gives smaller numbers higher priority. `build` copies its input and repairs internal nodes from the last parent up to the root; leaves already satisfy the heap rule.

In [ ]:
class MinHeap {
    constructor(compare = (a, b) => a - b) {
        this.heap = [];
        this.compare = compare;
    }

    size() { return this.heap.length; }
    isEmpty() { return this.size() === 0; }
    peek() { return this.heap[0]; }

    swap(a, b) {
        [this.heap[a], this.heap[b]] = [this.heap[b], this.heap[a]];
    }

    insert(value) {
        if (value == null) return false;
        this.heap.push(value);
        this.siftUp(this.size() - 1);
        return true;
    }

    siftUp(index) {
        while (index > 0) {
            const parent = Math.floor((index - 1) / 2);
            if (this.compare(this.heap[index], this.heap[parent]) >= 0) break;
            this.swap(index, parent);
            index = parent;
        }
    }

    siftDown(index) {
        while (true) {
            const left = 2 * index + 1;
            const right = 2 * index + 2;
            let best = index;
            if (left < this.size() && this.compare(this.heap[left], this.heap[best]) < 0) {
                best = left;
            }
            if (right < this.size() && this.compare(this.heap[right], this.heap[best]) < 0) {
                best = right;
            }
            if (best === index) break;
            this.swap(index, best);
            index = best;
        }
    }

    extract() {
        if (this.isEmpty()) return undefined;
        const root = this.heap[0];
        const last = this.heap.pop();
        if (!this.isEmpty()) {
            this.heap[0] = last;
            this.siftDown(0);
        }
        return root;
    }

    build(values) {
        this.heap = values.slice();
        for (let i = Math.floor(this.size() / 2) - 1; i >= 0; i--) {
            this.siftDown(i);
        }
        return this;
    }
}

**Try it in Notebook**

Create a heap and insert `5`, `3`, `8`, and `1`. Predict `peek()` and `size()` before running the code.

In [ ]:
const insertionExampleHeap = new MinHeap();
[5, 3, 8, 1].forEach(value => insertionExampleHeap.insert(value));
console.log('peek():', insertionExampleHeap.peek());
console.log('size():', insertionExampleHeap.size());

### Insertion, peek, and extraction

`peek` reads the root in **O(1)** time. To extract it, save the root, move the last element into its place, and sift down. At each step choose the higher-priority child (the smaller child in a min heap), so the current node can be no greater than either child. Extraction is **O(log n)**; extracting from an empty heap returns `undefined`.

In [ ]:
const lessonHeap = new MinHeap();
[5, 3, 8, 1].forEach(value => lessonHeap.insert(value));
console.log('peek and size:', lessonHeap.peek(), lessonHeap.size());
console.log('extraction order:', [
    lessonHeap.extract(), lessonHeap.extract(),
    lessonHeap.extract(), lessonHeap.extract()
]);
console.log('empty extraction:', lessonHeap.extract());

**Try it in Notebook**

Extract from `[2, 4, 3, 9, 7, 8]`. State the returned value and the array after repair.

In [ ]:
const extractionTryHeap = new MinHeap().build([2, 4, 3, 9, 7, 8]);
const extractionTryResult = extractionTryHeap.extract();
console.log('Returned value:', extractionTryResult);
console.log('Heap after repair:', extractionTryHeap.heap);


## 5. Reverse priority for a max heap

A max heap uses the same operations with reversed comparator arguments. This keeps one implementation of the shape and repair logic while allowing either priority direction.

In [ ]:
class MaxHeap extends MinHeap {
    constructor(compare = (a, b) => a - b) {
        super((a, b) => compare(b, a));
    }
}

const lessonMaxHeap = new MaxHeap();
[2, 5, 1, 8].forEach(value => lessonMaxHeap.insert(value));
console.log('maximum first:', lessonMaxHeap.extract());

**Try it in Notebook**

Starting with all four inserted values, predict the complete extraction order. What does one further `extract()` return?

In [ ]:
const maxHeapTryExample = new MaxHeap();
[2, 5, 1, 8].forEach(value => maxHeapTryExample.insert(value));
const maxHeapTryOrder = [];
while (!maxHeapTryExample.isEmpty()) {
    maxHeapTryOrder.push(maxHeapTryExample.extract());
}
const maxHeapTryExtra = maxHeapTryExample.extract();
console.log('Extraction order:', maxHeapTryOrder);
console.log('One further extraction:', maxHeapTryExtra);

## 6. Heap construction and heap sort

Bottom-up heap construction takes **O(n)** time: most nodes are near the leaves and can move only a short distance. For ascending heap sort, build a max heap, swap the maximum root with the end of the active range, shrink that range, then sift down the replacement root. The sorted suffix must not be included in later repairs. Heap sort takes **O(n log n)** time, uses **O(1)** extra space in this implementation, modifies its input, and is not stable.

In [ ]:
function heapSort(array) {
    function siftDown(index, size) {
        while (true) {
            let largest = index;
            const left = 2 * index + 1;
            const right = 2 * index + 2;
            if (left < size && array[left] > array[largest]) largest = left;
            if (right < size && array[right] > array[largest]) largest = right;
            if (largest === index) return;
            [array[index], array[largest]] = [array[largest], array[index]];
            index = largest;
        }
    }

    for (let i = Math.floor(array.length / 2) - 1; i >= 0; i--) {
        siftDown(i, array.length);
    }
    for (let end = array.length - 1; end > 0; end--) {
        [array[0], array[end]] = [array[end], array[0]];
        siftDown(0, end);
    }
    return array;
}

const valuesToSort = [8, 3, 5, 1, 9, 2, 7];
console.log(heapSort(valuesToSort));

## 7. Example: a task priority queue

A comparator can prioritize records rather than numbers. This queue runs the task with the smallest priority number first. Tasks with equal priority have no guaranteed relative order.

In [ ]:
const taskQueue = new MinHeap((a, b) => a.priority - b.priority);
[
    { name: 'Fix login', priority: 1 },
    { name: 'Update docs', priority: 3 },
    { name: 'Review code', priority: 2 },
    { name: 'Archive logs', priority: 4 }
].forEach(task => taskQueue.insert(task));
const scheduledTasks = [];
while (!taskQueue.isEmpty()) scheduledTasks.push(taskQueue.extract().name);
console.log(scheduledTasks);

## 8. Example: select the top-k retrieval candidates

A search system can score candidate passages for a question, then use a heap to select the strongest matches. The scores below are illustrative and are not generated by an AI model. For a large stream, retain only `k` items in a min heap: its root is the weakest selected score, which is the one to replace when a better candidate arrives. This takes **O(n log k)** time and **O(k)** heap space for `k > 0`.

In [ ]:
function topKByScore(items, k) {
    if (!Number.isInteger(k) || k < 0) {
        throw new RangeError('k must be a nonnegative integer');
    }
    const retained = new MinHeap((a, b) => a.score - b.score);
    for (const item of items) {
        if (retained.size() < k) {
            retained.insert(item);
        } else if (k > 0 && item.score > retained.peek().score) {
            retained.extract();
            retained.insert(item);
        }
    }
    return retained.heap.slice().sort((a, b) => b.score - a.score);
}

const retrievalCandidates = [
    { title: 'Reset your password', score: 0.91 },
    { title: 'Account security', score: 0.76 },
    { title: 'Recover account access', score: 0.84 },
    { title: 'Change email address', score: 0.63 }
];
console.log(topKByScore(retrievalCandidates, 2).map(item => item.title));

**Try it in Notebook**

Add `Password reset troubleshooting` with score `0.96` and keep `k = 2`. Predict the selected titles. Why does a bounded top-k heap use a min heap even though we want the highest scores?

In [ ]:
const candidatesWithTroubleshooting = [
    ...retrievalCandidates,
    { title: 'Password reset troubleshooting', score: 0.96 }
];
const boundedTopKHeap = new MinHeap((a, b) => a.score - b.score);
for (const candidate of candidatesWithTroubleshooting) {
    if (boundedTopKHeap.size() < 2) {
        boundedTopKHeap.insert(candidate);
    } else if (candidate.score > boundedTopKHeap.peek().score) {
        const replaced = boundedTopKHeap.extract();
        console.log('Replace weakest retained candidate:', replaced.title);
        boundedTopKHeap.insert(candidate);
    }
}
const boundedTopKResults = boundedTopKHeap.heap
    .slice()
    .sort((a, b) => b.score - a.score)
    .map(candidate => candidate.title);
console.log('Selected titles:', boundedTopKResults);
console.log('The min-heap root is the weakest retained score, so it is quick to replace.');

## 9. Exercises

Try these before opening the answers below.

1. Is `[2, 6, 3, 5]` a min heap? Which parent-child pair determines the answer?
2. In an array of length 7, find the parent and child indexes of index 2. Do the children of index 3 exist?
3. Insert `2` into `[3, 5, 4, 9, 7]`. Write the array after appending and after each swap.
4. Extract from `[2, 4, 3, 9, 7, 8]`. What value is returned and what is the repaired array?
5. What is the extraction order of a max heap after inserting `2, 5, 1, 8`?
6. Add a candidate with score `0.96` to the retrieval example and keep `k = 2`. Which titles are selected, in order?

### Exercise answers

1. No. The parent `6` is greater than its child `5`.
2. Index 2 has parent 0 and children 5 and 6. Index 3's children would be 7 and 8, so neither exists.
3. `[3, 5, 4, 9, 7, 2]` → `[3, 5, 2, 9, 7, 4]` → `[2, 5, 3, 9, 7, 4]`.
4. It returns `2`; after repair the heap is `[3, 4, 8, 9, 7]`.
5. `8, 5, 2, 1`; one more extraction returns `undefined`.
6. `Password reset troubleshooting`, then `Reset your password`. A min heap keeps the least competitive retained score at its root.

## Worked solutions for every Try it

Run the next cell to see each Try it worked through with the heap operations. For the index exercise, the complete-tree layout is compact because each node's parent and children are determined by its position; no pointers or gaps need to be stored. For insertion, each snapshot shows the append and every sift-up swap. For extraction, the snapshot shows the root returned and the repaired heap.

In [ ]:
(() => {
    const assert = require('node:assert/strict');

    // Try it 1: [2, 6, 3, 5] fails because 6 is greater than its child 5.
    const isMinHeapArray = values => values.every((value, index) => {
        const left = 2 * index + 1;
        const right = 2 * index + 2;
        return (left >= values.length || value <= values[left]) &&
            (right >= values.length || value <= values[right]);
    });
    const notAHeap = [2, 6, 3, 5];
    assert.equal(isMinHeapArray(notAHeap), false);
    console.log('1. Min heap?', isMinHeapArray(notAHeap),
        '- parent 6 at index 1 is greater than child 5 at index 3.');

    // Try it 2: index formulas and bounds for an array of length seven.
    const index = 2;
    const indexAnswer = {
        parent: Math.floor((index - 1) / 2),
        children: [2 * index + 1, 2 * index + 2],
        index3Children: [2 * 3 + 1, 2 * 3 + 2],
        index3ChildrenExist: [2 * 3 + 1, 2 * 3 + 2].map(child => child < 7)
    };
    assert.deepEqual(indexAnswer, {
        parent: 0, children: [5, 6], index3Children: [7, 8],
        index3ChildrenExist: [false, false]
    });
    console.log('2. Indexes:', indexAnswer);

    // Try it 3: append 2, then record each sift-up step.
    const insertionSteps = [[3, 5, 4, 9, 7]];
    const insertionHeap = new MinHeap();
    insertionHeap.heap = insertionSteps[0].slice();
    insertionHeap.heap.push(2);
    insertionSteps.push(insertionHeap.heap.slice());
    let insertionIndex = insertionHeap.size() - 1;
    while (insertionIndex > 0) {
        const parent = Math.floor((insertionIndex - 1) / 2);
        if (insertionHeap.heap[insertionIndex] >= insertionHeap.heap[parent]) break;
        insertionHeap.swap(insertionIndex, parent);
        insertionSteps.push(insertionHeap.heap.slice());
        insertionIndex = parent;
    }
    assert.deepEqual(insertionSteps, [
        [3, 5, 4, 9, 7], [3, 5, 4, 9, 7, 2],
        [3, 5, 2, 9, 7, 4], [2, 5, 3, 9, 7, 4]
    ]);
    console.log('3. Insert 2:', insertionSteps);

    // Try it 4: extract the root and inspect the heap after sift-down.
    const extractionHeap = new MinHeap().build([2, 4, 3, 9, 7, 8]);
    const extractedValue = extractionHeap.extract();
    assert.equal(extractedValue, 2);
    assert.deepEqual(extractionHeap.heap, [3, 4, 8, 9, 7]);
    console.log('4. Extract:', extractedValue, 'repaired heap:', extractionHeap.heap);

    // Try it 5: peek and size after inserting 5, 3, 8, and 1.
    const insertedHeap = new MinHeap();
    [5, 3, 8, 1].forEach(value => insertedHeap.insert(value));
    assert.equal(insertedHeap.peek(), 1);
    assert.equal(insertedHeap.size(), 4);
    console.log('5. Peek and size:', insertedHeap.peek(), insertedHeap.size());

    // Try it 6: extract all values from a max heap, then extract once more.
    const maxHeapTry = new MaxHeap();
    [2, 5, 1, 8].forEach(value => maxHeapTry.insert(value));
    const maxHeapOrder = [
        maxHeapTry.extract(), maxHeapTry.extract(),
        maxHeapTry.extract(), maxHeapTry.extract()
    ];
    const extraMaxExtract = maxHeapTry.extract();
    assert.deepEqual(maxHeapOrder, [8, 5, 2, 1]);
    assert.equal(extraMaxExtract, undefined);
    console.log('6. Max heap extraction order:', maxHeapOrder,
        'one more extraction:', extraMaxExtract);

    // Try it 7: add the 0.96 candidate and keep the two highest scores.
    const tryItTopK = topKByScore([
        ...retrievalCandidates,
        { title: 'Password reset troubleshooting', score: 0.96 }
    ], 2).map(item => item.title);
    assert.deepEqual(tryItTopK,
        ['Password reset troubleshooting', 'Reset your password']);
    console.log('7. Top two candidates:', tryItTopK);

    console.log('All Try it solutions passed.');
})();

## 10. Checks and review

The final cell checks heap structure as well as the visible examples. If you change an implementation, rerun the notebook from the top and use these assertions to catch regressions.

In [ ]:
(() => {
    const assert = require('node:assert/strict');
    const isMinHeap = heap => {
        for (let child = 1; child < heap.length; child++) {
            const parent = Math.floor((child - 1) / 2);
            if (heap[parent] > heap[child]) return false;
        }
        return true;
    };

    const checkHeap = new MinHeap();
    [5, 3, 8, 1, 2, 7].forEach(value => {
        checkHeap.insert(value);
        assert.equal(isMinHeap(checkHeap.heap), true);
    });
    assert.equal(checkHeap.peek(), 1);
    assert.deepEqual(
        [checkHeap.extract(), checkHeap.extract(), checkHeap.extract(),
            checkHeap.extract(), checkHeap.extract(), checkHeap.extract()],
        [1, 2, 3, 5, 7, 8]
    );
    assert.equal(checkHeap.extract(), undefined);
    assert.equal(new MinHeap().extract(), undefined);
    const singleValueHeap = new MinHeap();
    singleValueHeap.insert(42);
    assert.equal(singleValueHeap.extract(), 42);
    assert.equal(singleValueHeap.isEmpty(), true);
    const builtHeap = new MinHeap().build([9, 4, 7, 1, -2, 6]);
    assert.equal(isMinHeap(builtHeap.heap), true);
    assert.deepEqual(builtHeap.heap.slice().sort((a, b) => a - b), [-2, 1, 4, 6, 7, 9]);

    const checkMaxHeap = new MaxHeap();
    [2, 5, 1, 8].forEach(value => checkMaxHeap.insert(value));
    assert.deepEqual(
        [checkMaxHeap.extract(), checkMaxHeap.extract(),
            checkMaxHeap.extract(), checkMaxHeap.extract()],
        [8, 5, 2, 1]
    );
    assert.deepEqual(heapSort([8, 3, 5, 1, 9, 2, 7]), [1, 2, 3, 5, 7, 8, 9]);
    assert.deepEqual(heapSort([]), []);
    assert.deepEqual(heapSort([4]), [4]);
    assert.deepEqual(scheduledTasks, ['Fix login', 'Review code', 'Update docs', 'Archive logs']);
    assert.deepEqual(
        topKByScore(retrievalCandidates, 2).map(item => item.title),
        ['Reset your password', 'Recover account access']
    );
    assert.deepEqual(topKByScore(retrievalCandidates, 0), []);
    assert.throws(() => topKByScore(retrievalCandidates, -1), RangeError);
    assert.deepEqual(
        topKByScore([
            ...retrievalCandidates,
            { title: 'Password reset troubleshooting', score: 0.96 }
        ], 2).map(item => item.title),
        ['Password reset troubleshooting', 'Reset your password']
    );

    console.log('All Chapter 11 checks passed.');
})();

## Review

A heap combines a complete-tree shape with a local parent-child ordering rule. Insertion repairs upward; extraction replaces the root and repairs downward. The array representation makes parent and child access constant-time, while the tree's logarithmic height bounds those repairs. Bottom-up construction, heap sort, priority queues, and bounded top-k selection all reuse the same heap operations.